# 02 - Nettoyage des données

On corrige les problèmes trouvés dans `01_exploration` :

1. Supprimer le doublon
2. Remettre le zéro des codes postaux
3. Supprimer la colonne `Country`
4. Vérifier les `Product ID` qui ont plusieurs noms
5. Examiner les profits très négatifs

In [25]:
import pandas as pd

ventes = pd.read_csv(
    "../data/raw/Sample - Superstore.csv",
    encoding="latin-1",
    dtype={"Postal Code": str},
    parse_dates=["Order Date", "Ship Date"],
    date_format="%m/%d/%Y",
)
ventes.shape

(9994, 21)

In [26]:
ventes = ventes.drop_duplicates(subset=ventes.columns.drop("Row ID"))
ventes.shape

(9993, 21)

In [27]:
ventes["Postal Code"] = ventes["Postal Code"].str.zfill(5)
ventes["Postal Code"].str.len().value_counts()

Postal Code
5    9993
Name: count, dtype: int64

In [28]:
noms_par_id = ventes.groupby("Product ID")["Product Name"].nunique()
print("Product ID avec plusieurs noms :", (noms_par_id > 1).sum())

Product ID avec plusieurs noms : 32


**Décision :** on garde ces 32 produits tels quels. Nos questions portent sur les catégories, les régions et les dates, pas sur les produits un par un, donc cette erreur ne change pas nos résultats.

In [29]:
print("Commandes à perte :", (ventes["Profit"] < 0).sum())
ventes.groupby(ventes["Discount"] > 0)["Profit"].mean()


Commandes à perte : 1870


Discount
False    66.900292
True     -6.656116
Name: Profit, dtype: float64

**Décision :** on garde les profits négatifs. Ce ne sont pas des erreurs : les 1 870 commandes à perte ont **toutes** une remise. Sans remise, le profit moyen est de +66,90 $, avec remise il tombe à -6,66 $. Ces pertes sont réelles et servent à répondre à la question 3 (les remises font-elles baisser le profit ?).

## Conclusion du nettoyage

| Problème | Action | Résultat |
|---|---|---|
| 1 doublon | supprimé | 9 994 → 9 993 lignes |
| 438 codes postaux à 4 chiffres | zéro remis devant | tous à 5 chiffres |
| `Country` toujours identique | colonne supprimée | 21 → 20 colonnes |
| 32 `Product ID` avec 2 noms | gardés | sans effet sur nos questions |
| 1 870 profits négatifs | gardés | vraies pertes liées aux remises |

Fichier propre : `data/processed/ventes_propres.csv` (9 993 lignes, 20 colonnes).